In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS pvdaq_catalog.silver;

USE CATALOG pvdaq_catalog;
USE SCHEMA silver;
SELECT current_catalog(), current_schema();

-- Creating Silver tables as the tables from the end of the Bronze stage
CREATE TABLE IF NOT EXISTS inverters AS SELECT * FROM pvdaq_catalog.bronze.inverters;
CREATE TABLE IF NOT EXISTS meters AS SELECT * FROM pvdaq_catalog.bronze.meters;
CREATE TABLE IF NOT EXISTS metrics AS SELECT * FROM pvdaq_catalog.bronze.metrics;
CREATE TABLE IF NOT EXISTS modules AS SELECT * FROM pvdaq_catalog.bronze.modules;
CREATE TABLE IF NOT EXISTS mount AS SELECT * FROM pvdaq_catalog.bronze.mount;
CREATE TABLE IF NOT EXISTS other_instruments AS SELECT * FROM pvdaq_catalog.bronze.other_instruments;
CREATE TABLE IF NOT EXISTS pvdata_2020_sample AS SELECT * FROM pvdaq_catalog.bronze.pvdata_2020_sample;
CREATE TABLE IF NOT EXISTS site AS SELECT * FROM pvdaq_catalog.bronze.site;
CREATE TABLE IF NOT EXISTS system AS SELECT * FROM pvdaq_catalog.bronze.system;

In [0]:
%sql
-- The schema of the time series data table is: system_id, year, month, day, measured_on, utc_measured_on, metric_id, value
-- We need to be able to join the time series data with metadata tables on the metric_id in order to calculate certain metrics for PV systems over time

-- Selecting row counts of metadata tables to inspect
SELECT 'pvdaq_site' AS table_name, COUNT(*) AS row_count FROM site
UNION ALL
SELECT 'pvdaq_system' AS table_name, COUNT(*) AS row_count FROM system
UNION ALL
SELECT 'pvdaq_inverters' AS table_name, COUNT(*) AS row_count FROM inverters
UNION ALL
SELECT 'pvdaq_meters' AS table_name, COUNT(*) AS row_count FROM meters
UNION ALL
SELECT 'pvdaq_modules' AS table_name, COUNT(*) AS row_count FROM modules
UNION ALL
SELECT 'pvdaq_mount' AS table_name, COUNT(*) AS row_count FROM mount
UNION ALL
SELECT 'pvdaq_other_instruments' AS table_name, COUNT(*) AS row_count FROM other_instruments
UNION ALL
SELECT 'pvdaq_metrics' AS table_name, COUNT(*) AS row_count FROM metrics
ORDER BY table_name;

-- The above query shows that there are 1,768 unique values of metric_id. The PVDAQ documentation explains that sensor_name is "referenced name produced by the instrumentation or tagged by array owner." metric_id is a system-specific sensor key.
-- It seems that the researchers and developers of this data lake helpfully performed some grouping of the sensor_names under the field common_name, which is defined as "a general grouping of sensor types (e.g. DC voltage, AC energy, POA irradiance)." 

SELECT DISTINCT common_name FROM metrics;

-- As can be seen from the above query, there are 38 distinct common_name strings. These are the useful measurements that can be compared across PV systems, such as AC power, POA irradiance, wind speed, etc.


In [0]:
%sql
-- The measurements needed to calculate performance ratio using the PVAnalytics package from pvlib are AC power, POA irradiance, ambient temperature, and wind speed.

-- Query only the needed metrics from the metrics metadata table
SELECT 
  system_id,
  metric_id,
  common_name,
  units
FROM metrics
WHERE common_name IN (  -- Select needed measurements
    'AC power', 
    'Irradiance POA', 
    'Temperature ambient', 
    'Wind speed'
  ) AND
  system_id IN (1332,1276,1239,1278,3);  -- Select system IDs only from the sample

  -- Systems 1332 and 3 only have one of the four needed measurements (AC power and Irradiance POA, respectively)

-- Look at all the common_name values associated with 1332 and 3 
SELECT 
  system_id, 
  metric_id, 
  common_name, 
  units 
FROM metrics 
WHERE system_id IN (1332, 3)
ORDER BY system_id, common_name;

-- Confirmed, they are missing the other needed measurements



In [0]:
%sql
-- Find all system_ids in pvdaq_metrics that have all 4 required metrics
SELECT system_id
FROM metrics
WHERE common_name IN (
  'AC power',
  'Irradiance POA',
  'Temperature ambient',
  'Wind speed'
)
GROUP BY system_id
HAVING COUNT(DISTINCT common_name) = 4
ORDER BY system_id;

-- This means 23 systems are eligible for the PVAnalytics NREL Performance Ratio Function. Going back to Bronze layer to ingest only these systems.